# Churn — FE + XGBoost + LightGBM (prod-ready demo)
We’re building a reusable preprocessing pipeline and logging results to MLflow: 
- Clean column plan (numeric vs low/high-card categorical)
- Custom features: interactions + domain signals (CLV proxy, bundles, contract length)
- Encoders: One-Hot (low-card), Target/Frequency (high-card)
- Stratified split, metrics beyond accuracy (macro-F1, PR-AUC, ROC-AUC)
- MLflow run per model + artifacts (confusion matrix, tuned threshold)

In [1]:
# 1) Imports & setup
import os, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (f1_score, average_precision_score, roc_auc_score,
                             precision_recall_curve, classification_report,
                             ConfusionMatrixDisplay)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# MLflow
try:
    import mlflow
    USE_MLFLOW = True
except ImportError:
    USE_MLFLOW = False
    print("MLflow not installed — logging disabled.")

# Encoders
try:
    from category_encoders import TargetEncoder
    HAS_TE = True
except ImportError:
    HAS_TE = False
    print("category_encoders not installed — falling back.")

# Boosted trees
HAS_XGB, HAS_LGBM = False, False
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except:
    print("XGBoost not installed.")
try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except:
    print("LightGBM not installed.")

SEED = 42
np.random.seed(SEED)


LightGBM not installed.


In [2]:
# 2) MLflow config
TRACKING_URI = "http://127.0.0.1:5001"
EXPERIMENT_NAME = "churn-fe-demo"

if USE_MLFLOW:
    mlflow.set_tracking_uri(TRACKING_URI)
    mlflow.set_experiment(EXPERIMENT_NAME)
    print("Tracking URI:", TRACKING_URI)
    print("Experiment:", EXPERIMENT_NAME)
else:
    print("MLflow disabled or not installed — skipping config.")

Tracking URI: http://127.0.0.1:5001
Experiment: churn-fe-demo


## dataset modes
- **simple (numeric-only)**: fast model benchmarking, no categorical encoders.
- **churn-like (mixed types)**: realistic FE demo with OHE + target encoding.
Flip `USE_SIMPLE` to switch.


In [3]:
# 3) Data synthesis (toggle simple vs churn-like)
USE_SIMPLE = False  # True = numeric-only make_classification

if USE_SIMPLE:
    X, y = make_classification(
        n_samples=5000,
        n_features=30,
        n_informative=12,
        n_redundant=5,
        n_classes=2,
        weights=[0.7,0.3],
        class_sep=1.0,
        flip_y=0.01,
        random_state=SEED
    )
    df = pd.DataFrame(X, columns=[f"f{i}" for i in range(X.shape[1])])
    df["churn"] = y
    numeric_cols = [c for c in df.columns if c != "churn"]
    low_card_cats, high_card_cats = [], []
else:
    n = 5000
    rng = np.random.default_rng(SEED)
    df = pd.DataFrame({
        "tenure": rng.integers(1, 60, size=n),
        "monthly_charges": rng.normal(60, 25, size=n).clip(5, 200),
        "total_calls": rng.integers(10, 800, size=n),
        "months_active": rng.integers(1, 60, size=n),
        "contract_type": rng.choice(["month_to_month","one_year","two_year"], size=n, p=[0.6,0.25,0.15]),
        "gender": rng.choice(["M","F"], size=n),
        "payment_method": rng.choice(["card","upi","bank","cash"], size=n),
        "city": rng.choice([f"C{i}" for i in range(120)], size=n),
        "plan_name": rng.choice([f"P{i}" for i in range(80)], size=n),
    })
    # churn signal
    logit = (-2.0 + 0.02*(200 - df["monthly_charges"])
                   - 0.03*df["tenure"]
                   + 0.001*df["total_calls"]
                   + 0.20*(df["contract_type"]=="month_to_month").astype(int))
    p0 = 1/(1+np.exp(-logit))
    df["churn"] = (rng.random(n) < p0).astype(int)
    # services
    tier = df["plan_name"].str.extract(r"P(\d+)", expand=False).astype(int) % 3
    df["svc_security"]  = (tier==0).astype(int) ^ (rng.random(n) < 0.05)
    df["svc_backup"]    = (tier==1).astype(int) ^ (rng.random(n) < 0.05)
    df["svc_streaming"] = (tier==2).astype(int) ^ (rng.random(n) < 0.05)
    df["services_count"] = df[["svc_security","svc_backup","svc_streaming"]].sum(axis=1)
    delta = (-0.15*(df["services_count"]>=2).astype(int)
             + 0.10*(df["contract_type"]=="month_to_month").astype(int))
    p = (p0*0.85 + 0.15) + delta.clip(-0.25,0.25)
    df["churn"] = (rng.random(n) < p.clip(0.05,0.95)).astype(int)

    numeric_cols = ["tenure","monthly_charges","total_calls","months_active"]
    low_card_cats = ["contract_type","gender","payment_method"]
    high_card_cats = ["city","plan_name"]

print(df.head(3))
print("Shape:", df.shape, "| Target rate:", round(df["churn"].mean(),3))


   tenure  monthly_charges  total_calls  months_active   contract_type gender  \
0       6        50.597395          751             29        two_year      F   
1      46        48.547955          710             29  month_to_month      M   
2      39        79.224147          503             38  month_to_month      M   

  payment_method  city plan_name  churn  svc_security  svc_backup  \
0           bank   C31       P77      1         False       False   
1           cash   C34       P24      1          True       False   
2           card  C114       P40      0          True        True   

   svc_streaming  services_count  
0           True               1  
1          False               1  
2          False               2  
Shape: (5000, 14) | Target rate: 0.703


## feature engineering intent
We encode business hypotheses:
- Month-to-month contracts churn more.
- More bundled services → lower churn.
- Tenure × monthly_charges ≈ lifetime value proxy.
- Usage intensity (avg calls per month) matters.
No target is used to create features → **leak-safe**.


In [4]:
# 4) Feature builder
from sklearn.base import BaseEstimator, TransformerMixin

class ChurnFeatureBuilder(BaseEstimator, TransformerMixin):
    def __init__(self, add_interactions=True, add_domain=True):
        self.add_interactions = add_interactions
        self.add_domain = add_domain

    def fit(self, X, y=None): return self

    def transform(self, X):
        df = X.copy()
        if self.add_interactions:
            df["tenure_x_monthly"] = df["tenure"] * df["monthly_charges"]
        if self.add_domain:
            df["total_cost_est"] = df["tenure"] * df["monthly_charges"]
            df["clv_simple"] = (df["months_active"]+1) * df["monthly_charges"]
            df["contract_len_months"] = df["contract_type"].map({
                "month_to_month": 1, "one_year": 12, "two_year": 24
            })
            df["has_bundle"] = (df[["svc_security","svc_backup","svc_streaming"]].sum(axis=1) >= 2).astype(int)
            df["charges_bucket"] = pd.cut(df["monthly_charges"], bins=[0,30,70,120,200],
                                          labels=["low","mid","high","very_high"])
            df["call_intensity_bucket"] = pd.cut(df["total_calls"]/df["months_active"].clip(1,None),
                                                 bins=[0,50,150,400,1000],
                                                 labels=["low","mid","high","very_high"])
        return df


## models & expectations
- **Logistic Regression** (scaled): interpretable baseline.
- **RandomForest** (no scaling): non-linear baseline.

We will compare:
- **macro-F1** (class-balanced)
- **PR-AUC** (minority-aware)
- **ROC-AUC** (separability)
Then we pick a **best-F1 threshold** from the PR curve.


In [5]:
# 5) Update column plan
engineered_numeric = ["tenure_x_monthly","total_cost_est","clv_simple"]
engineered_cats    = ["charges_bucket","call_intensity_bucket"]

# merge + de-dupe while preserving order
numeric_cols   = list(dict.fromkeys(list(numeric_cols) + engineered_numeric))
low_card_cats  = list(dict.fromkeys(list(low_card_cats) + engineered_cats))
# (high_card_cats unchanged)


In [6]:
# 6) Preprocessors & pipeline
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    # older sklearn
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
    
te  = TargetEncoder() if HAS_TE else "passthrough"

full_fe = Pipeline([
    ("build", ChurnFeatureBuilder(add_interactions=True, add_domain=True)),
    ("pre", ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("impute", SimpleImputer(strategy="median")),
                ("scale", StandardScaler())
            ]), numeric_cols),
            ("ohe", ohe, low_card_cats),
            ("te",  te,  high_card_cats),
        ],
        remainder="drop",
        verbose_feature_names_out=False
    )),
])


In [7]:
# 7) Split data
TARGET = "churn"
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)


In [8]:
# 8) Unified helpers

def best_f1_threshold(y_true, prob):
    # handle degenerate probs (all same) and empty thresholds
    prec, rec, th = precision_recall_curve(y_true, prob)
    # avoid division/NaN
    f1s = 2*(prec*rec) / (prec + rec + 1e-12)
    # thresholds array is len-1 vs prec/rec; guard empty/NaN
    if len(th) == 0 or np.all(~np.isfinite(f1s[:-1])):
        return 0.5  # safe fallback
    idx = int(np.nanargmax(f1s[:-1]))
    val = th[idx]
    if not np.isfinite(val):
        return 0.5
    return float(val)

def log_confmat_to_mlflow(name, y_true, prob, thr, show=False):
    # guard plotting/logging so MLflow run never fails
    try:
        fig = plt.figure()
        ConfusionMatrixDisplay.from_predictions(y_true, (prob >= thr).astype(int))
        plt.title(f"Confusion Matrix — {name} (tuned)")
        plt.tight_layout()
        if USE_MLFLOW:
            mlflow.log_figure(fig, f"plots/cm_{name}.png")
        if show:
            plt.show()
    except Exception as e:
        print(f"[WARN] CM logging skipped for {name}: {e}")
    finally:
        plt.close('all')

def eval_and_log(name, clf, X_train, X_test, y_train, y_test,
                 tags=None, log_feature_importance=False):
    clf.fit(X_train, y_train)
    prob = clf.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)
    thr  = best_f1_threshold(y_test, prob)
    tuned = (prob >= thr).astype(int)

    m_def = f1_score(y_test, pred, average="macro")
    prauc = average_precision_score(y_test, prob)
    roc   = roc_auc_score(y_test, prob)
    m_tun = f1_score(y_test, tuned, average="macro")

    print(f"\n=== {name} ===")
    print(classification_report(y_test, pred, digits=3))
    print(f"macro F1 (0.5)={m_def:.3f} | PR-AUC={prauc:.3f} | ROC-AUC={roc:.3f}")
    print(f"best-F1 thr={thr:.3f} | macro F1 (tuned)={m_tun:.3f}")

    if USE_MLFLOW:
        try:
            with mlflow.start_run(run_name=name):
                mlflow.log_metric("macro_f1_default", float(m_def))
                mlflow.log_metric("pr_auc", float(prauc))
                mlflow.log_metric("roc_auc", float(roc))
                mlflow.log_metric("threshold_best_f1", float(thr))
                mlflow.log_metric("macro_f1_tuned", float(m_tun))
                mlflow.set_tag("owner", "raghu")
                mlflow.set_tag("purpose", "feature_engineering_demo")
                mlflow.set_tag("model_family", name.split("_")[0])
                mlflow.set_tag("threshold_policy", "best_f1_from_PR")
                if tags:
                    for k, v in tags.items():
                        mlflow.set_tag(k, v)

                # artifacts
                log_confmat_to_mlflow(name, y_test, prob, thr, show=False)

                if log_feature_importance:
                    try:
                        est = clf[-1] if isinstance(clf, Pipeline) else clf
                        if hasattr(est, "feature_importances_"):
                            imps = est.feature_importances_
                            names = [f"f{i}" for i in range(len(imps))]
                            idx = np.argsort(imps)[::-1][:20]
                            fig = plt.figure()
                            plt.bar(range(len(idx)), imps[idx])
                            plt.xticks(range(len(idx)), [names[i] for i in idx],
                                       rotation=45, ha="right")
                            plt.title(f"Top-20 Feature Importances — {name}")
                            plt.tight_layout()
                            mlflow.log_figure(fig, f"plots/feat_importance_{name}.png")
                            plt.close(fig)
                    except Exception as e:
                        print(f"[WARN] FI logging skipped for {name}: {e}")
        except Exception as e:
            # swallow MLflow logging exceptions so the run won’t be marked failed
            print(f"[WARN] MLflow logging failed for {name}: {e}")

    return {"f1": m_def, "prauc": prauc, "roc": roc, "f1_tuned": m_tun, "thr": thr}



In [9]:
# 9) Baseline models
logreg_pipe = Pipeline([
    ("fe", full_fe),
    ("clf", LogisticRegression(max_iter=500, class_weight="balanced"))
])

rf_pipe = Pipeline([
    ("fe", full_fe),
    ("clf", RandomForestClassifier(
        n_estimators=200, max_depth=8, class_weight="balanced", random_state=SEED))
])

results = {}
results["logreg"] = eval_and_log("logreg_fe", logreg_pipe, X_train,X_test,y_train,y_test)
results["rf"]     = eval_and_log("rf_fe", rf_pipe, X_train,X_test,y_train,y_test,
                                 log_feature_importance=True)



=== logreg_fe ===
              precision    recall  f1-score   support

           0      0.411     0.603     0.489       297
           1      0.791     0.636     0.705       703

    accuracy                          0.626      1000
   macro avg      0.601     0.619     0.597      1000
weighted avg      0.678     0.626     0.641      1000

macro F1 (0.5)=0.597 | PR-AUC=0.823 | ROC-AUC=0.672
best-F1 thr=0.189 | macro F1 (tuned)=0.506
🏃 View run logreg_fe at: http://127.0.0.1:5001/#/experiments/4/runs/931d7b153c4b4c11a0bd2f2542a60ccd
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/4

=== rf_fe ===
              precision    recall  f1-score   support

           0      0.424     0.485     0.452       297
           1      0.768     0.721     0.744       703

    accuracy                          0.651      1000
   macro avg      0.596     0.603     0.598      1000
weighted avg      0.666     0.651     0.657      1000

macro F1 (0.5)=0.598 | PR-AUC=0.827 | ROC-AUC=0.667
best

In [10]:
# 10) Tree-friendly FE + imbalance weight
from sklearn.base import clone as sk_clone
fe_tree = sk_clone(full_fe)
try:
    fe_tree.set_params(pre__num__scale="passthrough")
except:
    pass

pos_ratio = (y_train==1).mean()
scale_pos_weight = (1-pos_ratio)/max(pos_ratio,1e-6)
print("scale_pos_weight:", round(scale_pos_weight,3))


scale_pos_weight: 0.422


In [11]:
# 11) XGBoost
if HAS_XGB:
    xgb = XGBClassifier(
        n_estimators=500, learning_rate=0.05,
        max_depth=6, subsample=0.8, colsample_bytree=0.8,
        objective="binary:logistic", eval_metric="logloss",
        tree_method="hist", n_jobs=-1,
        random_state=SEED, scale_pos_weight=scale_pos_weight
    )
    xgb_pipe = Pipeline([("fe",fe_tree),("xgb",xgb)])
    results["xgb"] = eval_and_log("xgb_fe", xgb_pipe, X_train,X_test,y_train,y_test,
                                  log_feature_importance=True,
                                  tags={"scale_pos_weight":round(scale_pos_weight,3)})
else:
    print("XGB not available.")



=== xgb_fe ===
              precision    recall  f1-score   support

           0      0.396     0.424     0.410       297
           1      0.749     0.727     0.738       703

    accuracy                          0.637      1000
   macro avg      0.573     0.576     0.574      1000
weighted avg      0.644     0.637     0.640      1000

macro F1 (0.5)=0.574 | PR-AUC=0.812 | ROC-AUC=0.642
best-F1 thr=0.130 | macro F1 (tuned)=0.469
🏃 View run xgb_fe at: http://127.0.0.1:5001/#/experiments/4/runs/fb6848d929f14f3691c0e712c95472ca
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/4


In [12]:
# 12) LightGBM
if HAS_LGBM:
    lgbm = LGBMClassifier(
        n_estimators=600, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, n_jobs=-1,
        scale_pos_weight=scale_pos_weight
    )
    lgbm_pipe = Pipeline([("fe",fe_tree),("lgbm",lgbm)])
    results["lgbm"] = eval_and_log("lgbm_fe", lgbm_pipe, X_train,X_test,y_train,y_test,
                                   log_feature_importance=True,
                                   tags={"scale_pos_weight":round(scale_pos_weight,3)})
else:
    print("LightGBM not available.")


LightGBM not available.


In [13]:
# 13) Scoreboard
for k,v in results.items():
    print(f"{k.upper():5s} | F1={v['f1']:.3f} | F1_tuned={v['f1_tuned']:.3f} | "
          f"PR-AUC={v['prauc']:.3f} | ROC-AUC={v['roc']:.3f}")
print("\nCompare runs in MLflow UI → Experiments → churn-fe-demo")


LOGREG | F1=0.597 | F1_tuned=0.506 | PR-AUC=0.823 | ROC-AUC=0.672
RF    | F1=0.598 | F1_tuned=0.474 | PR-AUC=0.827 | ROC-AUC=0.667
XGB   | F1=0.574 | F1_tuned=0.469 | PR-AUC=0.812 | ROC-AUC=0.642

Compare runs in MLflow UI → Experiments → churn-fe-demo


## conclusions
- On our churn-like synthetic data, both **LogReg** and **RF** land around macro-F1 ≈ 0.59, ROC-AUC mid-0.6s, PR-AUC ≈ 0.82.  
- Threshold tuning helps modestly because probabilities are not fully calibrated (esp. for RF).
- The **feature engineering** pattern is sane and reusable: column plan → builder → encoders → model.

## what I’d do next:
1. **Calibrate RF** with `CalibratedClassifierCV` (isotonic, cv=3) → re-tune threshold.
2. **Add XGBoost/LightGBM** using the *same* preprocessor; expect a lift in ROC-AUC.
3. **Hyperparam search** (small `RandomizedSearchCV`) for RF/XGB.
4. **Model Registry**: register the best model (e.g., `churn-fe-best`) with tags:
   - `owner=raghu`, `purpose=interview_demo`, `dataset=synthetic_churn_70_30`
   - `preprocessor=full_fe_v1`, `threshold=best_f1`, `calibrated=true/false`
5. Optional: log a **feature importance** bar chart to MLflow for the tree models.


